# Aufgabe 3_3_3 – Temperaturregelung mit P/PI/PID-Regler

## Situation

Der Raum aus Aufgabe 3_3_2 soll nun auf eine **gewünschte Temperatur geregelt** werden.  
Das Heizsystem wird als PID-Regler abgebildet.

## Modell-Aufbau

```
Step(w=20°C) ──▶ [Sum] ──▶ PID ──▶ PT1(τ=120, K=0.35) ──▶ Raumtemperatur
                   ↑                                              │
                   └──────────────── Rückführung ─────────────────┘
```

| Block | Parameter |
|-------|-----------|
| Sollwert | Sprung von 15°C auf 20°C bei t=300 min |
| Regler | P-Regler: `Kp=3`, `Ki=0`, `Kd=0` |
| Strecke | PT1: `tau=120`, `K=0.35` |

In [ ]:
import sys
sys.path.insert(0, '..')
from blockdiagram import Step, Sum, PID, PT1, Scope

In [ ]:
# P-Regler (Ausgangspunkt)
Kp = 3.0
Ki = 0.0
Kd = 0.0

w      = Step(t_step=300, initial=15, final=20)   # Sollwert-Sprung
e      = Sum()
regler = PID(Kp=Kp, Ki=Ki, Kd=Kd, source=e)
raum   = PT1(tau=120, K=0.35, source=regler, initial=15*0.35/0.35)

e.connect(w,    sign=+1)
e.connect(raum, sign=-1)

Scope(t_end=1500, ylabel="Temperatur [°C]", xlabel="Zeit [min]",
      title=f"Temperaturregelung: Kp={Kp}, Ki={Ki}, Kd={Kd}").run(
    Sollwert=w,
    Raumtemperatur=raum,
    Regelabweichung=e
)

## Aufgaben

**a)** Wird die Soll-Temperatur mit dem P-Regler erreicht? Gibt es eine bleibende Regelabweichung?

*Antwort:*

**b)** Was verändert sich bei größerem/kleinerem `Kp`?

**c)** Was müsste man ändern, um die Regelabweichung vollständig zu eliminieren?

In [ ]:
# PI-Regler: Ki > 0 eliminiert bleibende Regelabweichung
Kp = 3.0
Ki = 0.02  # ← anpassen
Kd = 0.0

w      = Step(t_step=300, initial=15, final=20)
e      = Sum()
regler = PID(Kp=Kp, Ki=Ki, Kd=Kd, source=e)
raum   = PT1(tau=120, K=0.35, source=regler, initial=15)
e.connect(w, +1)
e.connect(raum, -1)

Scope(t_end=1500, ylabel="Temperatur [°C]", xlabel="Zeit [min]",
      title=f"PI-Regler: Kp={Kp}, Ki={Ki}").run(
    Sollwert=w, Raumtemperatur=raum
)

## Ziegler-Nichols (Aufgabe 3_3_5)

Erhöhen Sie `Kp` schrittweise (bei `Ki=0`, `Kd=0`) bis das System dauerhaft schwingt → kritische Verstärkung $K_P^{krit}$.  
Lesen Sie die Schwingungsdauer $T^{krit}$ aus dem Scope ab.

| Regler | $K_P$ | $K_I = K_P / T_n$ | $K_D = K_P \cdot T_v$ |
|--------|-------|-------------------|-----------------------|
| P      | $0.5 \cdot K_P^{krit}$ | 0 | 0 |
| PI     | $0.45 \cdot K_P^{krit}$ | $K_P / (0.85 T^{krit})$ | 0 |
| PID    | $0.6 \cdot K_P^{krit}$ | $K_P / (0.5 T^{krit})$ | $K_P \cdot 0.12 T^{krit}$ |

In [ ]:
# Ziegler-Nichols: Stabilitätsgrenze suchen
# Erhöhen Sie Kp bis das System schwingt

Kp = 5.0   # ← schrittweise erhöhen
Ki = 0.0
Kd = 0.0

w      = Step(t_step=300, initial=15, final=20)
e      = Sum()
regler = PID(Kp=Kp, Ki=Ki, Kd=Kd, source=e)
raum   = PT1(tau=120, K=0.35, source=regler, initial=15)
e.connect(w, +1)
e.connect(raum, -1)

Scope(t_end=2000, ylabel="Temperatur [°C]", xlabel="Zeit [min]",
      title=f"Stabilitätsgrenze suchen – Kp={Kp}").run(
    Sollwert=w, Raumtemperatur=raum
)

In [ ]:
# Ziegler-Nichols PID-Parameter anwenden
# Tragen Sie hier Ihre ermittelten Werte ein:
Kp_krit = 10.0  # ← gefundene kritische Verstärkung
T_krit  = 480.0 # ← abgelesene Schwingungsdauer in Minuten

# PID nach Ziegler-Nichols
Kp = 0.6  * Kp_krit
Tn = 0.5  * T_krit
Tv = 0.12 * T_krit
Ki = Kp / Tn
Kd = Kp * Tv
print(f"Kp={Kp:.3f}, Ki={Ki:.5f}, Kd={Kd:.1f}")

w      = Step(t_step=300, initial=15, final=20)
e      = Sum()
regler = PID(Kp=Kp, Ki=Ki, Kd=Kd, source=e)
raum   = PT1(tau=120, K=0.35, source=regler, initial=15)
e.connect(w, +1)
e.connect(raum, -1)

Scope(t_end=2000, ylabel="Temperatur [°C]", xlabel="Zeit [min]",
      title=f"PID nach Ziegler-Nichols").run(
    Sollwert=w, Raumtemperatur=raum
)